# Quantum Simulator — End-to-End Diagnosis
Fresh Colab workflow: tests → local circuit showcase → program-fault localization → synthetic hardware diagnosis → report export → optional real IBM execution last.

**Evidence labels matter:** synthetic hardware is not IBM hardware. No IBM job is submitted unless you explicitly enable the final cell.

In [ ]:
import os, shutil, subprocess, sys, json, time, platform
from pathlib import Path
REPO_URL='https://github.com/ruckscode/Quantum_Simulator.git'
BRANCH='fix/statistics-module-collision'
REPO=Path('/content/Quantum_Simulator')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO)],check=True)
os.chdir(REPO)
print('Branch:',subprocess.check_output(['git','branch','--show-current'],text=True).strip())
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

## 1. Install only core dependencies and run tests
This stage does not install Qiskit, use credentials, or contact IBM. Skipped tests are not successful benchmark runs.

In [ ]:
%pip install -e . pytest

In [ ]:
test_start=time.time()
test_run=subprocess.run([sys.executable,'-m','pytest','-q','--import-mode=importlib'],text=True,capture_output=True)
print(test_run.stdout)
if test_run.stderr: print(test_run.stderr[-4000:])
print('Exit code:',test_run.returncode,'Duration (s):',round(time.time()-test_start,2))

## Local acceptance report

Run the repository's official offline acceptance script as well. It writes its own report and explicitly marks Qiskit/IBM availability; it does not submit a live job.


In [ ]:
acceptance_run=subprocess.run([sys.executable,'clean_simulation_test.py'],text=True,capture_output=True)
print(acceptance_run.stdout)
if acceptance_run.stderr: print(acceptance_run.stderr[-4000:])
print('Acceptance exit code:',acceptance_run.returncode)
acceptance_path=Path('reports/clean_simulation_test_report.json')
acceptance_report=json.loads(acceptance_path.read_text()) if acceptance_path.exists() else {}
print('Saved report:',acceptance_path)
print('Real IBM execution:',acceptance_report.get('real_ibm_hardware_executed',False))


## 2. Run circuits and display counts
Finite-shot results naturally fluctuate. A circuit passing here only validates that specific circuit and simulator path.

In [ ]:
from quantum_simulator import QuantumCircuit
from quantum_simulator.hardware_model import HardwareModel
from quantum_simulator.diagnosis_engine import diagnose_execution
from datetime import datetime, timezone
SHOTS=2048; SEED=42
def build_circuits():
    out={}
    c=QuantumCircuit(1); c.h(0); c.measure_all(); out['hadamard_1q']=c
    c=QuantumCircuit(1); c.x(0); c.measure_all(); out['pauli_x_1q']=c
    c=QuantumCircuit(2); c.h(0); c.cx(0,1); c.measure_all(); out['bell_state']=c
    c=QuantumCircuit(3); c.h(0); c.cx(0,1); c.cx(1,2); c.measure_all(); out['ghz_3q']=c
    c=QuantumCircuit(3); c.x(0); c.x(1); c.ccx(0,1,2); c.measure_all(); out['toffoli_3q']=c
    return out
circuits=build_circuits(); circuit_records=[]
for name,circuit in circuits.items():
    try:
        counts=circuit.run(shots=SHOTS,seed=SEED).get_counts()
        row={'circuit':name,'status':'PASS','qubits':circuit.n_qubits,'operations':circuit.instructions,'shots':SHOTS,'counts':counts,'error':None}
    except Exception as e:
        row={'circuit':name,'status':'FAIL','qubits':circuit.n_qubits,'operations':repr(circuit.instructions),'shots':SHOTS,'counts':{},'error':f'{type(e).__name__}: {e}'}
    circuit_records.append(row); print(f'\n=== {name}: {row["status"]} ==='); print('Operations:',row['operations']); print('Counts:',row['counts']); print('Error:',row['error'])
print('\nCircuit total:',sum(r['status']=='PASS' for r in circuit_records),'/',len(circuit_records),'passed')

## 3. Diagnose healthy execution, injected program fault, and synthetic hardware anomaly
These are controlled fixtures for demonstration—not Bugs4Q/QMutBench scores and not physical hardware evidence.

In [ ]:
healthy=diagnose_execution(expected_distribution={'00':0.5,'11':0.5},observed_distribution={'00':0.5,'11':0.5},expected_circuit=[('h',0),('cx',0,1),('measure',[0,1])],observed_circuit=[('h',0),('cx',0,1),('measure',[0,1])],shots=SHOTS,threshold=0.05)
program_fault=diagnose_execution(expected_distribution={'0':0.0,'1':1.0},observed_distribution={'0':0.5,'1':0.5},expected_circuit=[('x',0),('measure',0,[0])],observed_circuit=[('h',0),('measure',0,[0])],shots=SHOTS,threshold=0.05)
synthetic_model=HardwareModel(1,backend_name='SYNTHETIC_FIXTURE'); synthetic_model.set_qubit_properties(0,readout_error=0.40)
hardware_case=diagnose_execution(expected_distribution={'0':1.0,'1':0.0},observed_distribution={'0':0.60,'1':0.40},expected_circuit=[('measure',0,[0])],observed_circuit=[('measure',0,[0])],hardware_model=synthetic_model,shots=SHOTS,threshold=0.05)
diagnoses=[]
for label,result,source in [('healthy_bell',healthy,'controlled local fixture'),('injected_program_fault',program_fault,'controlled local fixture'),('synthetic_hardware_anomaly',hardware_case,'synthetic calibration fixture')]:
    stats=result.statistics_result
    row={'case':label,'source':source,'category':result.category.value,'anomaly_detected':result.anomaly_detected,'summary':result.summary,'tvd':getattr(stats,'total_variation_distance',None) if stats else None,'threshold':getattr(stats,'threshold',None) if stats else None,'program_candidates':[str(x) for x in result.program_evidence_candidates],'hardware_candidates':[str(x) for x in result.hardware_evidence_candidates],'evidence':list(result.evidence),'missing_evidence':list(result.missing_evidence)}
    diagnoses.append(row); print('\n===',label,'==='); print(json.dumps(row,indent=2,default=str))
if program_fault.program_result:
    print('\nProgram localization candidates:')
    for candidate in program_fault.program_result.candidates: print(candidate)

## Full five-scenario diagnostic report

The standalone demo additionally covers ambiguous and insufficient-evidence cases. These are controlled offline fixtures, not benchmark ground truth or physical hardware results.


In [ ]:
demo_run=subprocess.run([sys.executable,'demo.py','--output','reports/colab_five_scenario_demo.json'],text=True,capture_output=True)
print(demo_run.stdout)
if demo_run.stderr: print(demo_run.stderr[-4000:])
print('Demo exit code:',demo_run.returncode)
demo_path=Path('reports/colab_five_scenario_demo.json')
demo_report=json.loads(demo_path.read_text()) if demo_path.exists() else None
if demo_report is None: raise RuntimeError('Five-scenario demo report was not created')
import pandas as pd
from IPython.display import display
summary=[]
for s in demo_report['scenarios']:
    summary.append({'Scenario':s['scenario_name'],'Expected':s['expected_category'],'Detected':s['actual_category'],'Pass':s['pass'],'Program candidates':len(s['program_evidence']['candidates']),'Hardware candidates':len(s['hardware_evidence']['candidates'])})
display(pd.DataFrame(summary))
for s in demo_report['scenarios']:
    print('\n',s['scenario_name'],'| detected:',s['actual_category'])
    print('Expected circuit:',s['execution_metadata']['expected_circuit'])
    print('Observed circuit:',s['execution_metadata']['observed_circuit'])
    print('Program candidates:',s['program_evidence']['candidates'])
    print('Hardware candidates:',s['hardware_evidence']['candidates'])
    print('Evidence:',s['diagnosis']['evidence'])


## 4. Benchmark audit and limits

The earlier audit was polluted by generated report/documentation files being discovered as dataset inputs; that discovery bug is fixed and regression-tested. Re-run the benchmark report on this checkout before relying on new counts. A report file is not raw benchmark data, and no accuracy/F1 claim is valid without independent, diagnosis-compatible ground truth.


### Optional full benchmark evaluation

The full QMutBench data can be large and may take a long time. This is opt-in and separate from the presentation demo. Keep disabled until the raw data is present and runtime is available.


In [ ]:
RUN_FULL_BENCHMARKS=False
if RUN_FULL_BENCHMARKS:
    benchmark_run=subprocess.run([sys.executable,'benchmark_validation.py','--root','.','--report-directory','reports/benchmarks'],text=True,capture_output=True)
    print(benchmark_run.stdout[-12000:])
    if benchmark_run.stderr: print(benchmark_run.stderr[-6000:])
    print('Benchmark exit code:',benchmark_run.returncode)
else:
    print('Full benchmark skipped by design. Set RUN_FULL_BENCHMARKS=True when raw datasets and runtime budget are ready.')


In [ ]:
Path('reports').mkdir(exist_ok=True)
report={'created_at_utc':datetime.now(timezone.utc).isoformat(),'branch':subprocess.check_output(['git','branch','--show-current'],text=True).strip(),'commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'python':platform.python_version(),'test_exit_code':test_run.returncode,'test_output':test_run.stdout,'circuits':circuit_records,'diagnoses':diagnoses,'acceptance_exit_code':acceptance_run.returncode,'acceptance_report':str(acceptance_path),'full_demo_report':str(demo_path),'full_demo_passed':demo_report['all_passed'],'benchmark_run_requested':RUN_FULL_BENCHMARKS,'real_ibm_hardware_executed':False,'ibm_status':'NOT_ATTEMPTED','benchmark_metric_policy':'No predictive performance metrics without independent compatible ground truth.'}
Path('reports/colab_end_to_end_diagnosis_report.json').write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
import csv
with open('reports/colab_end_to_end_summary.csv','w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['record_type','name','status','category','details']); writer.writeheader()
    for r in circuit_records: writer.writerow({'record_type':'circuit','name':r['circuit'],'status':r['status'],'category':'','details':json.dumps(r,default=str)})
    for r in diagnoses: writer.writerow({'record_type':'diagnosis','name':r['case'],'status':'RECORDED','category':r['category'],'details':json.dumps(r,default=str)})
print('Saved reports/colab_end_to_end_diagnosis_report.json and reports/colab_end_to_end_summary.csv')
print('Test exit code:',test_run.returncode,'; acceptance exit code:',acceptance_run.returncode,'; circuit passes:',sum(r['status']=='PASS' for r in circuit_records),'/',len(circuit_records)); print('Five-scenario demo passed:',demo_report['all_passed']); print('Real IBM hardware executed: False')


## 5. FINAL STAGE ONLY — real IBM hardware (explicit opt-in)
This cell is disabled by default. Before enabling it, make sure local tests and reports are satisfactory. Add your IBM token in Colab Secrets as `QISKIT_IBM_TOKEN`, set an exact backend name and the instance/CRN if required, then change `SUBMIT_TO_IBM` to `True`. Submitting creates a real job and may consume account quota. Never print or paste the token.

In [ ]:
SUBMIT_TO_IBM=False  # Explicitly change to True only when ready to submit a real job.
IBM_BACKEND_NAME=''  # Exact backend accessible to your account.
IBM_INSTANCE=''      # Optional instance/CRN if your account requires it.
IBM_SHOTS=2048
if not SUBMIT_TO_IBM:
    print('IBM submission skipped: no token read, no backend queried, no job submitted.')
else:
    from google.colab import userdata
    token=userdata.get('QISKIT_IBM_TOKEN')
    if not token: raise RuntimeError('Add QISKIT_IBM_TOKEN to Colab Secrets. Do not paste it into output/chat.')
    if not IBM_BACKEND_NAME.strip(): raise RuntimeError('Set IBM_BACKEND_NAME to an exact accessible backend.')
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qiskit", "qiskit-ibm-runtime"], check=True)
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit import QuantumCircuit as QiskitCircuit, transpile
    kwargs={'channel':'ibm_quantum_platform','token':token}
    if IBM_INSTANCE.strip(): kwargs['instance']=IBM_INSTANCE.strip()
    service=QiskitRuntimeService(**kwargs); backend=service.backend(IBM_BACKEND_NAME.strip())
    print('Backend:',backend.name,'Qubits:',getattr(backend,'num_qubits','unknown'))
    qc=QiskitCircuit(2); qc.h(0); qc.cx(0,1); qc.measure_all()
    isa=transpile(qc,backend=backend,optimization_level=1)
    job=Sampler(backend).run([isa],shots=IBM_SHOTS)
    print('Submitted real IBM job:',job.job_id())
    result=job.result(); counts=result[0].data.meas.get_counts()
    print('REAL_IBM_EXECUTED counts:',counts)
    print('Compare these counts with the local Bell counts above. A single difference does not by itself prove a hardware fault; correlate with calibration and repeat runs.')

## Completion checklist
- Review all test failures and skips; skips are not passes.
- Inspect each circuit's operations and counts.
- Inspect the injected program-fault localization evidence.
- Keep synthetic anomaly output clearly labelled as synthetic.
- Download/save the JSON and CSV reports.
- Only then enable the IBM cell to compare real backend counts and calibration evidence.